In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI

import numpy as np

load_dotenv('../.env')

client = OpenAI()


reference = "Qual'è il tuo animale domestico preferito?"

def get_embedding(txt):
    response = client.embeddings.create(
        input = txt,
        model = "text-embedding-3-small"
    )
    return response.data[0].embedding


def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

reference_embedding = get_embedding(reference)
document_embedding = get_embedding("Non mi piacciono i broccoli a colazione")
similarity = cosine_similarity(reference_embedding, document_embedding)

print("Cosine Similiarity: ", similarity)


Cosine Similiarity:  0.22902148088931912


In [ ]:
from chromadb.utils import embedding_functions
from dotenv import load_dotenv 
import chromadb



openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key="inserire env",
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    embedding_function=openai_ef,
    name="libri"
)

collection.add(
    documents=[
        "Informazioni legate alla programmazione software",
        "Contenuto legato agli strumenti AI",
        "Libro per commercialisti, sulla partita doppia",
        "Libro sul trading delle opzioni",
        "Libro sui bitcoin"
    ],
    metadatas=[
        {"source": "document", "page": 1},
        {"source": "document", "page": 2},
        {"source": "web site", "page": 993},
        {"source": "book", "page": 994},
        {"source": "document", "page": 12},
    ],
    ids=[
        "id_1",
        "id_2",
        "id_3",
        "id_4",
        "id_5",
    ]
)

results = collection.query(
    query_texts=["Voglio comprare bitcoin"],
    n_results=1
)

results


{'ids': [['id_5']],
 'embeddings': None,
 'documents': [['Libro sui bitcoin']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'page': 12, 'source': 'document'}]],
 'distances': [[0.43310052156448364]]}

In [28]:

collection.update(
    ids=["id_1"],
    documents=["Qui si parla di marketing"],
    metadatas=[{"source": "ebook", "page": 15, "lines":"14-36"}]
)

results = collection.query(
    query_texts="Come si vende un prodotto?",
    n_results=1
)

results


{'ids': [['id_1']],
 'embeddings': None,
 'documents': [['Qui si parla di marketing']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'page': 15, 'source': 'ebook', 'lines': '14-36'}]],
 'distances': [[0.5579188466072083]]}

In [ ]:
collection.delete(ids = ['id_1'])

results = collection.query(
    query_texts="Come si vende un prodotto?",
    n_results=1
)

results

# Collections disponibili
chroma_client.list_collections()

# ritorna i dati della collections
collection.get()

# ritorna i dati della collections con param
collection.get()['documents']


['Contenuto legato agli strumenti AI',
 'Libro per commercialisti, sulla partita doppia',
 'Libro sul trading delle opzioni',
 'Libro sui bitcoin']